# Notebook 2 — Historical Dataset

Explores and validates the multi-year dataset built by `pipeline/build_historical_dataset.py`.

**Each row** = one company in one fiscal year.  
**Features** = all fraud signals + value metrics computed from that year's 10-K.  
**Label** = 12-month forward stock return starting from the 10-K filing date.

This notebook checks:
1. Dataset shape, coverage, missing data
2. ROI distribution and outliers
3. Feature distributions over time (drift check)
4. Look-ahead bias audit
5. Signal vs ROI preliminary correlations

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('dark_background')

# ── Detect environment ────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    import subprocess
    subprocess.run(['wget', '-q', '-O', 'historical_dataset.parquet',
        'https://raw.githubusercontent.com/sherlock718/stock-fraud-screener/main/data/historical_dataset.parquet'])
    parquet_path = 'historical_dataset.parquet'
else:
    parquet_path = '../data/historical_dataset.parquet'

df = pd.read_parquet(parquet_path)
print(f"Rows: {len(df):,} | Companies: {df['ticker'].nunique():,} | Columns: {len(df.columns)}")
df.head(3)

## 1. Dataset shape & coverage

In [ ]:
# Rows per fiscal year
print("Rows per fiscal year:")
print(df['fiscal_year'].value_counts().sort_index())
print(f"\nForward returns available: {df['forward_return_12m'].notna().sum():,} / {len(df):,}")
print(f"Beat S&P 500 labels: {df['beat_sp500'].notna().sum():,}")

In [ ]:
# Feature coverage
feature_cols = [
    'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'non_op_ratio',
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'gross_margin', 'net_margin', 'debt_to_equity', 'current_ratio',
    'pe_ratio', 'pb_ratio', 'ev_ebitda', 'ncav_ratio'
]
existing = [c for c in feature_cols if c in df.columns]
cov = df[existing].notna().mean().sort_values(ascending=False)
cov_df = cov.reset_index()
cov_df.columns = ['feature', 'coverage']
cov_df['coverage_%'] = (cov_df['coverage'] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(cov_df['feature'], cov_df['coverage'], color='#3498db')
ax.axvline(0.7, color='red', linestyle='--', alpha=0.5, label='70% threshold')
ax.set_xlabel('Coverage (fraction of rows)')
ax.set_title('Feature Coverage in Historical Dataset')
ax.legend()
plt.tight_layout()
plt.show()
print(cov_df.to_string(index=False))

## 2. ROI distribution

In [ ]:
roi = df['forward_return_12m'].dropna()
roi_clipped = roi.clip(-1, 3)  # clip extreme outliers for display

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(roi_clipped, bins=80, color='#3498db', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='white', linestyle='--', alpha=0.5)
axes[0].axvline(roi.median(), color='#2ecc71', linestyle='-', label=f'Median {roi.median():.1%}')
axes[0].set_title('12-Month Forward Return Distribution')
axes[0].set_xlabel('Return (clipped at -100% / +300%)')
axes[0].legend()

# By year
roi_by_year = df.groupby('fiscal_year')['forward_return_12m'].median()
axes[1].bar(roi_by_year.index, roi_by_year.values, color='#f39c12', alpha=0.8)
axes[1].axhline(0, color='white', linestyle='--', alpha=0.5)
axes[1].set_title('Median 12-Month Return by Fiscal Year')
axes[1].set_xlabel('Fiscal Year')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

print(f"Count: {len(roi):,}")
print(f"Mean:   {roi.mean():.1%}")
print(f"Median: {roi.median():.1%}")
print(f"Std:    {roi.std():.1%}")
print(f">0:     {(roi > 0).mean():.1%} of companies had positive 12m return")
print(f"Beat S&P: {df['beat_sp500'].mean():.1%} of companies beat the S&P 500")

## 3. Look-ahead bias audit

Critical: we must verify that no future information leaked into the features.
The filing date must be BEFORE the entry price date.

In [ ]:
# Check: filed_date should always be <= entry price date (which is filed_date + 0-10 days)
# Check: no fiscal_year data should use prices from after that year ended

has_both = df[df['forward_return_12m'].notna() & df['filed_date'].notna()].copy()
has_both['filed_ts'] = pd.to_datetime(has_both['filed_date'])
has_both['year_end'] = has_both['fiscal_year'].apply(lambda y: pd.Timestamp(f'{y}-12-31'))

# Filing should be after the fiscal year ended (10-K filed typically 60-90 days after FY end)
early_filings = has_both[has_both['filed_ts'] < has_both['year_end']]
print(f"Total rows with forward return: {len(has_both):,}")
print(f"Rows where filing is BEFORE fiscal year end (potential bias): {len(early_filings):,}")
if len(early_filings) > 0:
    print("\nSample early filings (check these):")
    print(early_filings[['ticker', 'fiscal_year', 'filed_date', 'year_end']].head(10))
else:
    print("✅ No look-ahead bias detected — all filings are after fiscal year end")

## 4. Feature distributions over time

Check for data drift — do the features look stable across years?

In [ ]:
drift_features = ['beneish_score', 'altman_score', 'earnings_yield', 'gross_profitability', 'fcf_yield']
drift_features = [f for f in drift_features if f in df.columns]

fig, axes = plt.subplots(1, len(drift_features), figsize=(4*len(drift_features), 4))
if len(drift_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, drift_features):
    by_year = df.groupby('fiscal_year')[feat].median()
    ax.plot(by_year.index, by_year.values, marker='o', color='#3498db')
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('FY')

plt.suptitle('Median Feature Value by Fiscal Year (drift check)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Signal vs ROI — preliminary correlations

Which features are most correlated with 12-month forward return?

In [ ]:
label = 'forward_return_12m'
roi_df = df[df[label].notna()].copy()
roi_df[label] = roi_df[label].clip(-1, 3)  # winsorise extremes

# Correlation with ROI
feature_cols = [
    'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'non_op_ratio',
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'gross_margin', 'net_margin', 'debt_to_equity', 'current_ratio',
    'pe_ratio', 'pb_ratio', 'ev_ebitda', 'ncav_ratio',
]
existing = [c for c in feature_cols if c in roi_df.columns]

corrs = roi_df[existing + [label]].corr()[label].drop(label).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors, alpha=0.8)
ax.axvline(0, color='white', alpha=0.3)
ax.set_title(f'Pearson Correlation with 12m Forward Return (n={len(roi_df):,})')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

## 6. Fraud score vs forward return

Key hypothesis: high fraud score → lower future returns

In [ ]:
from pipeline.score_and_report import composite_score, risk_label
import sys
sys.path.insert(0, '..')

# Recompute fraud score for historical rows (signals are already in df)
# Map each row's signals into the format composite_score expects
def row_to_signals(row):
    return {
        'beneish':             {'score': row.get('beneish_score'), 'manipulator': row.get('beneish_flag', False)},
        'piotroski':           {'score': row.get('piotroski_score'), 'weak': row.get('piotroski_weak', False)},
        'accruals':            {'ratio': row.get('accruals_ratio'), 'red_flag': row.get('accruals_flag', False)},
        'cash_flow_divergence':{'divergence': row.get('cfd_ratio'), 'red_flag': row.get('cfd_flag', False)},
        'altman':              {'score': row.get('altman_score'), 'zone': row.get('altman_zone'), 'distress': row.get('altman_flag', False)},
        'revenue_quality':     {'ar_ratio': row.get('ar_ratio'), 'dso': row.get('dso'), 'red_flag': row.get('revenue_quality_flag', False)},
        'earnings_quality':    {'non_op_ratio': row.get('non_op_ratio'), 'red_flag': row.get('earnings_quality_flag', False)},
        'going_concern':       {'flagged': row.get('going_concern_flag', False)},
        'auditor':             {'auditor_name': None},
        'market':              {'avg_volume': None},
        'insider':             {'net_insider_shares': None},
    }

df['fraud_score'] = df.apply(lambda r: composite_score(row_to_signals(r)), axis=1)

# Bucket into fraud quartiles
roi_df2 = df[df['forward_return_12m'].notna() & df['fraud_score'].notna()].copy()
roi_df2['fraud_quartile'] = pd.qcut(roi_df2['fraud_score'], q=4,
                                     labels=['Q1 Low Risk', 'Q2', 'Q3', 'Q4 High Risk'])

avg_ret = roi_df2.groupby('fraud_quartile', observed=True)['forward_return_12m'].agg(['mean', 'median', 'count'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(avg_ret.index, avg_ret['median'], color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'], alpha=0.85)
ax.axhline(0, color='white', linestyle='--', alpha=0.4)
ax.set_title('Median 12m Forward Return by Fraud Score Quartile')
ax.set_xlabel('Fraud Score Quartile')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

print(avg_ret)